In [0]:
print ("Day 3 - DataBricks Class")

### Delta Table and Features

In [0]:
%sql
-- sample managed table
CREATE TABLE IF NOT EXISTS sample_managed_table (
  id INT,
  name STRING,
  country STRING
)
;

In [0]:
%sql
-- sample external table.
CREATE TABLE sample_external_table (
  id INT,
  name STRING,
  country STRING
)
LOCATION 's3 bucket';

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, lit
from datetime import date
# Define schema explicitly
schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("join_date",  DateType(),    True),
])
# Sample data
data = [
    (101, "Arjun",    "Engineering", "Chennai",   85000.0, date(2020, 3, 15)),
    (102, "Priya",    "Marketing",   "Mumbai",    72000.0, date(2019, 7, 1)),
    (103, "Karthik",  "Engineering", "Bangalore", 91000.0, date(2021, 1, 10)),
    (104, "Sneha",    "HR",          "Chennai",   65000.0, date(2018, 11, 20)),
    (105, "Rahul",    "Finance",     "Delhi",     78000.0, date(2022, 5, 5)),
    (106, "Divya",    "Engineering", "Bangalore", 88000.0, date(2020, 9, 3)),
    (107, "Manoj",    "Marketing",   "Mumbai",    70000.0, date(2023, 2, 14)),
    (108, "Anitha",   "HR",          "Chennai",   63000.0, date(2017, 8, 25)),
]
df = spark.createDataFrame(data, schema)
df.show()
target = "employeeproject.default.employee"
# Also register as a SQL table for SQL-style queries
spark.sql(f"""DROP TABLE IF EXISTS {target}""")
df.write.mode("overwrite").saveAsTable(f"""{target}""")
print("Sample Delta table created successfully!")

In [0]:
# -----------------------------------------------------------------------------
# 1A. Schema Enforcement — Delta rejects writes with mismatched schema
# -----------------------------------------------------------------------------
print("\n--- 1A. Schema Enforcement ---")
# Bad data: extra column 'bonus' not in original schema
bad_data = [(109, "Vikram", "Sales", "Pune", 74000.0, date(2023, 6, 1), 5000.0)]
bad_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("city",       StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("join_date",  DateType(),    True),
    StructField("bonus",      DoubleType(),  True),   # 🆕 Extra column
])
bad_df = spark.createDataFrame(bad_data, bad_schema)
try:
    bad_df.write.mode("append").saveAsTable(f"""{target}""")
except Exception as e:
    print(f"❌ Schema Enforcement Error (Expected): {e}")
    print("✅ Delta blocked the write because 'bonus' column doesn't exist in the table schema.")

In [0]:
# -----------------------------------------------------------------------------
# 1B. Schema Evolution — Allow new columns using mergeSchema
# -----------------------------------------------------------------------------
print("\n--- 1B. Schema Evolution (mergeSchema) ---")
bad_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"""{target}""")
evolved_df = spark.read.table(f"""{target}""")
evolved_df.show()
print("Schema evolved: 'bonus' column added. Existing rows show NULL for the new column.")